# Notebook 06 — Publish the trained models on Hugging Face

**Requires:** Notebook 02 finished (recipe-P weights in `06_models/`) and Notebook 05 finished (ConvNeXt-Base recipe-B weights). **No training.**
**Runtime:** T4 GPU recommended (the check in step 3 takes ≈ 5 min on a GPU, ≈ 30–40 min on a CPU).

### What this notebook does
| Step | What | Why |
|---|---|---|
| 2 | Lists every model in `06_models/` with its results (`04_results/`, or `04_results_rerun/` for the model re-trained in Notebook 05) | You see exactly what will be published |
| 3 | **Re-tests every published fold model on its own test fold** and compares the predictions with the ones recorded when it was trained | Proves the uploaded weights are the weights behind the reported numbers |
| 4 | Converts the weights to `.safetensors` (the safe Hugging Face format), and writes a `config.json` per model, the split files, `inference.py` and the model card (`README.md`) | Anyone can load a model in a few lines |
| 5 | Checks that `inference.py` loads the converted files and gives the same predictions | The usage code in the model card is tested |
| 6 | Uploads everything to one Hugging Face model repository and saves an upload record on Drive | Gives the link for the manuscript (Code availability) |

### Before you run it (once)
1. Create a free account at **huggingface.co** (if you do not have one).
2. Create a token: **huggingface.co → Settings → Access Tokens → Create new token → type "Write"** → copy it.
3. In Colab, click the **key icon (Secrets)** in the left sidebar → **Add new secret** → Name `HF_TOKEN`, Value = the token →
   switch **Notebook access** on. (Never paste the token into a code cell.)

Then **Runtime → Run all**. Re-running is safe: the repository is updated, not duplicated.

**When finished:** send me the printed summary of steps 3 and 6 (or `09_logs/hf_upload_NB06.json`).

## 0. Settings

In [ ]:
import os
NOTEBOOK, NB_TAG = "06_Publish_Models_HuggingFace", "NB06"
DRIVE_ROOT = os.environ.get("PEANUT_DRIVE_ROOT", "/content/drive/MyDrive/Peanut_Revision")
LOCAL_ROOT = os.environ.get("PEANUT_LOCAL_ROOT", "/content/peanut")

HF_REPO_NAME = "peanut-leaf-disease-classifiers"   # becomes https://huggingface.co/<your user name>/<this name>
HF_PRIVATE   = True      # True: only you can see it. Make it public (Settings -> Change visibility) after checking.
HF_LICENSE   = "cc-by-4.0"   # licence of the weights (same as the dataset)
VERIFY       = True      # step 3: re-test every fold model (strongly recommended)
UPLOAD       = True      # step 6: upload to Hugging Face

# Filled in later (the model card is updated when you run the notebook again):
GITHUB_URL   = ""        # e.g. "https://github.com/<user>/peanut-leaf-disease"
ZENODO_DOI   = ""        # e.g. "10.5281/zenodo.1234567"
PAPER_TITLE  = ("Deep learning-based peanut leaf disease classification using transfer learning "
                "under leakage-controlled cross-validation")
DATASET_DOI  = "10.17632/x6x5jkk873.2"

TEST_MODE = os.environ.get("PEANUT_TEST") == "1"   # smoke test outside Colab only
if TEST_MODE:
    UPLOAD = False

## 1. Drive and environment

In [ ]:
!pip -q install -U huggingface_hub safetensors
try:
    from google.colab import drive
    drive.mount("/content/drive"); IN_COLAB = True
except ImportError:
    IN_COLAB = False

import json, time, shutil, zipfile, hashlib, platform, datetime as dt
from pathlib import Path
from dataclasses import dataclass, asdict, replace
import numpy as np, pandas as pd
import torch, torch.nn as nn, torchvision
import torchvision.transforms as T
import torchvision.models as M
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import safetensors, huggingface_hub
from safetensors.torch import save_file, load_file

D = {k: Path(DRIVE_ROOT) / v for k, v in {"data": "01_data", "splits": "02_splits", "results": "04_results",
     "models": "06_models", "logs": "09_logs"}.items()}
for k in ("splits", "results", "models"):
    if not D[k].exists():
        raise SystemExit(f"{D[k]} not found. Check DRIVE_ROOT and that Notebooks 01 and 02 have finished.")
D["logs"].mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
GPU = torch.cuda.get_device_name(0) if AMP else "CPU"
if not AMP and VERIFY and not TEST_MODE:
    print("No GPU: step 3 will run on the CPU (about 30-40 min). For speed: Runtime -> Change runtime type -> T4 GPU.")

ENV = {"notebook": NOTEBOOK, "run_at": dt.datetime.now().isoformat(timespec="seconds"), "gpu": GPU,
       "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
       "safetensors": safetensors.__version__, "huggingface_hub": huggingface_hub.__version__}
print(json.dumps(ENV, indent=2))

CLASSES = ["Alternaria leaf spot", "Healthy", "Leaf spot (early and late)", "Rosette", "Rust"]
NUM_WORKERS = 0 if TEST_MODE else 2

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

## 2. What will be published

In [ ]:
@dataclass
class Config:
    name: str
    backbone: str
    recipe: str
    aug: str = "A"
    sampler: bool = True
    class_weighted_loss: bool = True
    label_smoothing: float = 0.0
    optimizer: str = "adam"
    lr: float = 1e-3
    weight_decay: float = 1e-5
    scheduler: str = "onecycle"
    epochs: int = 25
    finetune: str = "partial"
    head: str = "mlp"
    batch_size: int = 32
    img_size: int = 224
    select_metric: str = "val_macro_f1"
    pretrained: bool = True
    split: str = "cv_splits"          # which split file in 02_splits/ (cv_splits = group-aware)
    notes: str = ""

def recipe_P(backbone, **kw):
    big = backbone == "efficientnet_b4"
    kw.setdefault("name", f"{backbone}__P")
    return Config(backbone=backbone, recipe="P", img_size=380 if big else 224, batch_size=16 if big else 32, **kw)

def recipe_B(backbone, **kw):
    big = backbone == "efficientnet_b4"
    kw.setdefault("name", f"{backbone}__B")
    return Config(backbone=backbone, recipe="B", aug="B", sampler=False, class_weighted_loss=False,
                  label_smoothing=0.2, optimizer="adamw", lr=1e-5, weight_decay=1e-4, scheduler="cosine_wr",
                  finetune="full", head="linear", img_size=380 if big else 224, batch_size=16 if big else 32, **kw)

BACKBONES = ["resnet50", "efficientnet_b0", "efficientnet_b4", "convnext_base"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def get_transforms(aug, size):
    if aug == "A":    # Disease_CNN_Transfer_learning_RasNet_50.ipynb
        train = T.Compose([T.Resize((size, size)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
                           T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
                           T.RandomAffine(degrees=0, translate=(0.1, 0.1)), T.ToTensor(), T.RandomErasing(p=0.2),
                           T.Normalize(MEAN, STD)])
    elif aug == "B":  # Disease-EfficientNet-B0/B4, Disease-ConvNeXt-Base
        train = T.Compose([T.RandomHorizontalFlip(p=0.5), T.RandomRotation(15), T.RandomResizedCrop(size, scale=(0.7, 1.0)),
                           T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
                           T.RandomAffine(degrees=0, translate=(0.1, 0.1)), T.RandomGrayscale(p=0.1), T.ToTensor(),
                           T.Normalize(MEAN, STD)])
    elif aug == "none":
        train = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    else:
        raise ValueError(aug)
    test = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    return train, test

def build_model(cfg, num_classes=5):
    def head(in_f):
        if cfg.head == "mlp":
            return nn.Sequential(nn.Linear(in_f, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes))
        return nn.Linear(in_f, num_classes)
    b, pre = cfg.backbone, cfg.pretrained
    if b == "resnet50":
        m = M.resnet50(weights=M.ResNet50_Weights.IMAGENET1K_V1 if pre else None)
        m.fc = head(m.fc.in_features); trainable = ["layer4.", "fc."]
    elif b == "efficientnet_b0":
        m = M.efficientnet_b0(weights=M.EfficientNet_B0_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[1] = head(m.classifier[1].in_features); trainable = ["features.6.", "features.7.", "features.8.", "classifier."]
    elif b == "efficientnet_b4":
        m = M.efficientnet_b4(weights=M.EfficientNet_B4_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[1] = head(m.classifier[1].in_features); trainable = ["features.6.", "features.7.", "features.8.", "classifier."]
    elif b == "convnext_base":
        m = M.convnext_base(weights=M.ConvNeXt_Base_Weights.IMAGENET1K_V1 if pre else None)
        m.classifier[2] = head(m.classifier[2].in_features); trainable = ["features.6.", "features.7.", "classifier."]
    else:
        raise ValueError(b)
    if cfg.finetune == "partial":
        for n, p in m.named_parameters():
            p.requires_grad = any(n.startswith(t) for t in trainable)
    return m

cv = pd.read_csv(D["splits"] / "cv_splits.csv")
idx = pd.read_csv(D["splits"] / "index_original.csv")
cv_meta = json.loads((D["splits"] / "cv_splits_meta.json").read_text())
sig_file = D["results"] / "split_signature.json"
if sig_file.exists() and json.loads(sig_file.read_text()) != cv_meta:
    raise SystemExit("04_results/ was produced with different splits than 02_splits/. Stop and tell me.")

MODELS_TO_PUBLISH = {}   # name -> {"cfg": dict, "folds": {fold: {...}}}
rows = []
for mdir in sorted(p for p in D["models"].iterdir() if p.is_dir()):
    name = mdir.name
    rerun_dir = Path(DRIVE_ROOT) / "04_results_rerun"
    src = rerun_dir if (rerun_dir / name / "config.json").exists() else D["results"]   # re-trained in Notebook 05?
    cfg_file = src / name / "config.json"
    if not cfg_file.exists():
        print(f"skipped {name}: no results in 04_results/{name}"); continue
    cfg = json.loads(cfg_file.read_text())
    folds = {}
    for wf in sorted(mdir.glob("r0_f*.pt")):
        f = int(wf.stem.split("_f")[1])
        mfile = src / name / f"r0_f{f}" / "metrics.json"
        if not mfile.exists():
            print(f"skipped {name} fold {f}: weights without results"); continue
        m = json.loads(mfile.read_text())
        folds[f] = {"weights": wf, "metrics": m, "size_mb": wf.stat().st_size / 1e6}
    if folds:
        MODELS_TO_PUBLISH[name] = {"cfg": cfg, "folds": folds, "src": src, "rerun": src != D["results"]}
        acc = np.array([v["metrics"]["accuracy"] for v in folds.values()]) * 100
        rows.append({"model": name, "results from": src.name, "folds": len(folds), "accuracy (%) mean ± SD":
                     f"{acc.mean():.2f} ± {acc.std(ddof=1) if len(acc) > 1 else 0:.2f}",
                     "MB per fold": f"{np.mean([v['size_mb'] for v in folds.values()]):.0f}",
                     "MB total": f"{sum(v['size_mb'] for v in folds.values()):.0f}"})

INVENTORY = pd.DataFrame(rows)
display(INVENTORY)
n_expected = cv_meta["k"]
for name, v in MODELS_TO_PUBLISH.items():
    if len(v["folds"]) != n_expected:
        print(f"WARNING: {name} has {len(v['folds'])} of {n_expected} folds")
not_saved = sorted(p.name for p in D["results"].iterdir() if p.is_dir() and (p / "config.json").exists()
                   and p.name not in MODELS_TO_PUBLISH)
print("Configurations with results but no saved weights (not published):", ", ".join(not_saved) or "none")

## 3. Check: do the saved weights reproduce the reported predictions?
Every fold model is loaded and re-tested on **its own** test fold. Its predictions are compared with
`04_results/<model>/r0_f<k>/predictions.csv`, which all tables of the paper are computed from.
Agreement should be 100 % or very close to it (the weights are stored in half precision, so an image that was a
near-tie can rarely flip).

In [ ]:
VERIFICATION = {}
if VERIFY:
    local_cache = Path(LOCAL_ROOT) / "cache448"
    if not any(local_cache.rglob("*.jpg")):
        t0 = time.time()
        local_zip = Path(LOCAL_ROOT) / "groundnut_v2_cache448.zip"
        local_zip.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(D["data"] / "groundnut_v2_cache448.zip", local_zip)
        with zipfile.ZipFile(local_zip) as z:
            z.extractall(LOCAL_ROOT)
        local_zip.unlink()
        print(f"cache copied and unzipped in {time.time()-t0:.0f}s")
    IMAGES = {iid: np.asarray(Image.open(local_cache / rel).convert("RGB")) for iid, rel in zip(idx["image_id"], idx["cache_rel"])}
    LABELS = dict(zip(idx["image_id"], idx["label"]))

    class ImageSet(Dataset):
        def __init__(self, ids, tf): self.ids, self.tf = list(ids), tf
        def __len__(self): return len(self.ids)
        def __getitem__(self, i): return self.tf(Image.fromarray(IMAGES[self.ids[i]])), LABELS[self.ids[i]]

    @torch.no_grad()
    def predict(model, loader):
        model.eval(); ps, probs = [], []
        for x, _ in loader:
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=AMP):
                out = model(x.to(DEVICE, non_blocking=True))
            out = out.float(); probs.append(torch.softmax(out, 1).cpu().numpy()); ps.append(out.argmax(1).cpu().numpy())
        return np.concatenate(ps), np.concatenate(probs)

    vrows = []
    for name, v in MODELS_TO_PUBLISH.items():
        cfg = Config(**{**v["cfg"], "pretrained": False})
        _, tf_test = get_transforms(cfg.aug, cfg.img_size)
        for f, fv in v["folds"].items():
            s = cv[(cv["repeat"] == 0) & (cv["fold"] == f) & (cv["role"] == "test")]
            saved = pd.read_csv(v["src"] / name / f"r0_f{f}" / "predictions.csv")
            if saved["image_id"].tolist() != s["image_id"].tolist():
                raise SystemExit(f"{name} fold {f}: test images differ from the split file. Stop and tell me.")
            model = build_model(cfg)
            sd = torch.load(fv["weights"], map_location="cpu", weights_only=True)
            model.load_state_dict({k: (t.float() if t.is_floating_point() else t) for k, t in sd.items()})
            model.to(DEVICE)
            dl = DataLoader(ImageSet(s["image_id"], tf_test), batch_size=64, num_workers=NUM_WORKERS, pin_memory=AMP)
            p_now, prob_now = predict(model, dl)
            prob_saved = saved[[f"p_{i}" for i in range(len(CLASSES))]].to_numpy()
            y = saved["y_true"].to_numpy()
            res = {"n_test": int(len(y)),
                   "agreement": float((p_now == saved["y_pred"].to_numpy()).mean()),
                   "n_different": int((p_now != saved["y_pred"].to_numpy()).sum()),
                   "accuracy_reported": float((saved["y_pred"].to_numpy() == y).mean()),
                   "accuracy_now": float((p_now == y).mean()),
                   "max_prob_difference": float(np.abs(prob_now - prob_saved).max())}
            VERIFICATION.setdefault(name, {})[f] = res
            vrows.append({"model": name, "fold": f, "test images": res["n_test"],
                          "agreement (%)": f"{100*res['agreement']:.2f}", "different": res["n_different"],
                          "acc. recorded (%)": f"{100*res['accuracy_reported']:.2f}",
                          "acc. now (%)": f"{100*res['accuracy_now']:.2f}",
                          "max |Δp|": f"{res['max_prob_difference']:.4f}"})
            del model; torch.cuda.empty_cache() if AMP else None
        print(f"{name}: checked {len(v['folds'])} folds")
    VTABLE = pd.DataFrame(vrows); display(VTABLE)
    tot_diff = sum(r["n_different"] for d in VERIFICATION.values() for r in d.values())
    tot_n = sum(r["n_test"] for d in VERIFICATION.values() for r in d.values())
    print(f"\nOverall: {tot_n - tot_diff}/{tot_n} test predictions identical to the reported ones "
          f"({100*(tot_n - tot_diff)/tot_n:.2f}%).")
    if tot_diff > 0.005 * tot_n:
        print("WARNING: more than 0.5% of the predictions differ. Do not upload yet - send me this table.")
else:
    print("VERIFY = False: check skipped.")

## 4. Build the repository folder (safetensors, configs, split files, inference code, model card)

In [ ]:
INFERENCE_PY = r'''"""Load a model from this repository and classify peanut leaf images.

Example:
    python inference.py --model resnet50__P --fold 0 leaf1.jpg leaf2.jpg

Each configuration has five models, one per cross-validation fold. For new images any fold can be used
(or all five, averaging their probabilities with --fold all). To reproduce the paper, evaluate a fold model
only on its own test fold (role == "test" in cv_splits.csv); the other images were used to train it.
"""
import argparse
import json
from pathlib import Path

import torch
import torch.nn as nn
import torchvision.models as M
import torchvision.transforms as T
from PIL import Image, ImageOps
from safetensors.torch import load_file

HERE = Path(__file__).resolve().parent
CLASSES = ["Alternaria leaf spot", "Healthy", "Leaf spot (early and late)", "Rosette", "Rust"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def build_model(backbone, head, num_classes=len(CLASSES)):
    """Same architecture as in training (ImageNet weights are not needed: all weights are loaded from the file)."""
    def make_head(in_f):
        if head == "mlp":
            return nn.Sequential(nn.Linear(in_f, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes))
        return nn.Linear(in_f, num_classes)
    if backbone == "resnet50":
        m = M.resnet50(weights=None); m.fc = make_head(m.fc.in_features)
    elif backbone == "efficientnet_b0":
        m = M.efficientnet_b0(weights=None); m.classifier[1] = make_head(m.classifier[1].in_features)
    elif backbone == "efficientnet_b4":
        m = M.efficientnet_b4(weights=None); m.classifier[1] = make_head(m.classifier[1].in_features)
    elif backbone == "convnext_base":
        m = M.convnext_base(weights=None); m.classifier[2] = make_head(m.classifier[2].in_features)
    else:
        raise ValueError(backbone)
    return m


def load_model(model, fold, device="cpu", root=HERE):
    """model: folder name, e.g. "resnet50__P"; fold: 0-4. Returns (model in eval mode, config dict)."""
    cfg = json.loads((Path(root) / model / "config.json").read_text())
    net = build_model(cfg["training"]["backbone"], cfg["training"]["head"])
    sd = load_file(str(Path(root) / model / f"fold{fold}.safetensors"))
    net.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in sd.items()})
    return net.eval().to(device), cfg


def preprocess(image, img_size):
    """Same steps as in the paper: camera orientation, RGB, 448 x 448 (bicubic), then the network input size."""
    img = image if isinstance(image, Image.Image) else Image.open(image)
    img = ImageOps.exif_transpose(img).convert("RGB").resize((448, 448), Image.BICUBIC)
    return T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(MEAN, STD)])(img)


@torch.no_grad()
def predict_proba(net, cfg, images, device="cpu"):
    """Returns a tensor (n_images x 5) of class probabilities, columns in the order of CLASSES."""
    x = torch.stack([preprocess(im, cfg["img_size"]) for im in images]).to(device)
    return torch.softmax(net(x).float(), dim=1).cpu()


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", default="resnet50__P", help="folder name, e.g. resnet50__P")
    ap.add_argument("--fold", default="0", help="0-4, or 'all' to average the five fold models")
    ap.add_argument("images", nargs="+")
    a = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    folds = range(5) if a.fold == "all" else [int(a.fold)]
    prob = None
    for f in folds:
        net, cfg = load_model(a.model, f, dev)
        p = predict_proba(net, cfg, a.images, dev)
        prob = p if prob is None else prob + p
    prob = prob / len(folds)
    for path, p in zip(a.images, prob):
        k = int(p.argmax())
        print(f"{path}: {CLASSES[k]} ({100 * float(p[k]):.1f}%)")
'''

STAGE = Path(LOCAL_ROOT) / "hf_repo"
shutil.rmtree(STAGE, ignore_errors=True); STAGE.mkdir(parents=True)
NICE = {"resnet50": "ResNet-50", "efficientnet_b0": "EfficientNet-B0", "efficientnet_b4": "EfficientNet-B4",
        "convnext_base": "ConvNeXt-Base"}
FILES = {}
for name, v in MODELS_TO_PUBLISH.items():
    out = STAGE / name; out.mkdir()
    fold_info = {}
    for f, fv in v["folds"].items():
        sd = torch.load(fv["weights"], map_location="cpu", weights_only=True)
        sd = {k: t.contiguous() for k, t in sd.items()}
        fn = out / f"fold{f}.safetensors"
        save_file(sd, str(fn), metadata={"model": name, "fold": str(f), "repeat": "0", "format": "pt"})
        m = fv["metrics"]
        fold_info[str(f)] = {"file": fn.name, "sha256": sha256_file(fn), "best_epoch": m["best_epoch"],
                             "n_train": m["n_train"], "n_val": m["n_val"], "n_test": m["n_test"],
                             "test_accuracy": m["accuracy"], "test_macro_f1": m["macro_f1"],
                             "test_n_correct": m.get("n_correct"),
                             "check_agreement_with_reported_predictions": VERIFICATION.get(name, {}).get(f, {}).get("agreement")}
        FILES[f"{name}/{fn.name}"] = fold_info[str(f)]["sha256"]
    acc = [x["test_accuracy"] for x in fold_info.values()]; mf1 = [x["test_macro_f1"] for x in fold_info.values()]
    cfg = v["cfg"]
    card_cfg = {"name": name, "backbone": cfg["backbone"], "backbone_name": NICE.get(cfg["backbone"], cfg["backbone"]),
                "recipe": cfg["recipe"], "retrained_in_notebook_05": v["rerun"], "classes": CLASSES, "img_size": cfg["img_size"],
                "preprocessing": ["PIL ImageOps.exif_transpose", "convert RGB", "resize 448x448 bicubic",
                                  f"torchvision Resize(({cfg['img_size']}, {cfg['img_size']}))", "ToTensor",
                                  "Normalize(ImageNet mean, std)"],
                "normalisation": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
                "training": cfg, "weights_dtype": "float16",
                "selection": "epoch with the highest validation macro-F1 within each fold",
                "cv": {"type": "stratified group k-fold", "k": cv_meta["k"], "repeat": 0, "split_file": "cv_splits.csv"},
                "folds": fold_info,
                "summary": {"accuracy_mean": float(np.mean(acc)), "accuracy_sd": float(np.std(acc, ddof=1)) if len(acc) > 1 else 0.0,
                            "macro_f1_mean": float(np.mean(mf1)), "macro_f1_sd": float(np.std(mf1, ddof=1)) if len(mf1) > 1 else 0.0,
                            "pooled_correct": int(sum(x["test_n_correct"] or 0 for x in fold_info.values())),
                            "pooled_n": int(sum(x["n_test"] for x in fold_info.values()))}}
    (out / "config.json").write_text(json.dumps(card_cfg, indent=2))
    v["card_cfg"] = card_cfg
    print(f"{name}: {len(fold_info)} folds converted")

cv.to_csv(STAGE / "cv_splits.csv", index=False)
idx[["image_id", "class", "label", "src_rel", "sha256", "dup_group"]].rename(
    columns={"src_rel": "file_in_mendeley_archive", "dup_group": "duplicate_group"}).to_csv(STAGE / "images.csv", index=False)
(STAGE / "cv_splits_meta.json").write_text(json.dumps(cv_meta, indent=2))
(STAGE / "inference.py").write_text(INFERENCE_PY)
for fn in ["cv_splits.csv", "images.csv", "cv_splits_meta.json", "inference.py"]:
    FILES[fn] = sha256_file(STAGE / fn)
print("staged in", STAGE)

In [ ]:
def fmt(x): return f"{100*x:.2f}"
code_line = f"[{GITHUB_URL}]({GITHUB_URL})" if GITHUB_URL else "GitHub repository (link added on release)"
zen_line = f" · archived at Zenodo: [https://doi.org/{ZENODO_DOI}](https://doi.org/{ZENODO_DOI})" if ZENODO_DOI else ""

trows = []
for name, v in MODELS_TO_PUBLISH.items():
    c = v["card_cfg"]; s = c["summary"]
    dag = " †" if v["rerun"] else ""
    trows.append(f"| `{name}/`{dag} | {c['backbone_name']} | {c['recipe']} | {len(c['folds'])} | {c['img_size']} | "
                 f"{fmt(s['accuracy_mean'])} ± {fmt(s['accuracy_sd'])} | {fmt(s['macro_f1_mean'])} ± {fmt(s['macro_f1_sd'])} | "
                 f"{s['pooled_correct']} / {s['pooled_n']} |")
best = max(MODELS_TO_PUBLISH, key=lambda n: MODELS_TO_PUBLISH[n]["card_cfg"]["summary"]["accuracy_mean"])
example = "resnet50__P" if "resnet50__P" in MODELS_TO_PUBLISH else best
check_line = ""
if VERIFICATION:
    tot_diff = sum(r["n_different"] for d in VERIFICATION.values() for r in d.values())
    tot_n = sum(r["n_test"] for d in VERIFICATION.values() for r in d.values())
    check_line = (f"Before upload, every fold model was re-tested on its own test fold: {tot_n - tot_diff} of {tot_n} "
                  f"predictions were identical to the predictions recorded when the models were trained.")
rerun_notes = []
cmp_file = Path(DRIVE_ROOT) / "04_results_rerun" / "rerun_comparison.json"
CMP = json.loads(cmp_file.read_text()) if cmp_file.exists() else {}
for name, v in MODELS_TO_PUBLISH.items():
    if not v["rerun"]: continue
    s = v["card_cfg"]["summary"]; c = CMP.get(name, {})
    o = [json.loads(p.read_text()) for p in sorted((D["results"] / name).glob("r0_f*/metrics.json"))]
    oa = np.array([m["accuracy"] for m in o])
    txt = (f"† `{name}`: the weights of the run reported in the paper were not saved, so this configuration was trained "
           f"again with the same code, settings, folds and random seeds (Notebook 05). GPU training is not bit-for-bit "
           f"repeatable: the published run reached {fmt(s['accuracy_mean'])} ± {fmt(s['accuracy_sd'])}% "
           f"({s['pooled_correct']} / {s['pooled_n']} correct); the run reported in the paper reached "
           f"{fmt(oa.mean())} ± {fmt(np.std(oa, ddof=1) if len(oa) > 1 else 0)}% "
           f"({sum(m['n_correct'] for m in o)} / {sum(m['n'] for m in o)}).")
    if c:
        txt += (f" The two runs gave the same prediction for {c['n_same']} of {c['n']} test images "
                f"(McNemar exact test, p = {c['mcnemar_p']:.2g}).")
    rerun_notes.append(txt)
not_pub = [n for n in not_saved if "__naive_split" not in n and "-" not in n and n != "resnet50__vanilla_TL"]

CARD = f'''---
license: {HF_LICENSE}
library_name: pytorch
pipeline_tag: image-classification
tags:
- image-classification
- plant-disease
- peanut
- groundnut
- agriculture
- transfer-learning
- torchvision
metrics:
- accuracy
- f1
---

# Peanut leaf disease classifiers (leakage-controlled cross-validation)

Trained models from the study **"{PAPER_TITLE}"** (manuscript under review; citation below will be updated).

* Code: {code_line}{zen_line}
* Dataset: Sasmal *et al.*, groundnut leaf dataset, Mendeley Data, version 2, [https://doi.org/{DATASET_DOI}](https://doi.org/{DATASET_DOI}) (CC BY 4.0)

The models classify a close-up photograph of a peanut (groundnut, *Arachis hypogaea*) leaf into five classes:
**Alternaria leaf spot, Healthy, Leaf spot (early and late), Rosette, Rust**.

## What is in this repository

Each configuration was trained with stratified **group** 5-fold cross-validation: near-identical photographs of the
dataset were grouped, and a group never appears in more than one of the training, validation and test sets of a fold.
For every configuration there are **five models, one per fold** (`fold0`–`fold4`). Each was selected on its own
validation set (highest validation macro-F1) and tested once on its own test fold.

| Folder | Backbone | Recipe | Folds | Input (px) | Test accuracy (%), mean ± SD over folds | Macro-F1 (%), mean ± SD | Correct / test images (pooled) |
|---|---|---|---|---|---|---|---|
{chr(10).join(trows)}

Recipe P: last stage and a new two-layer head trained (rest frozen); balanced sampling, class-weighted cross-entropy,
Adam with OneCycle learning rate, flip/rotation/colour/translation/random-erasing augmentation.
Recipe B: all layers trained; linear head; label smoothing 0.2; AdamW with cosine warm restarts; flip/rotation/
random-resized-crop/colour/translation/greyscale augmentation. 25 epochs in both cases. Full settings are in each
`config.json` (`training`).
{(chr(10) + "Weights were not kept for: " + ", ".join(f"`{n}`" for n in not_pub) + ". Their results are reported in the paper and can be reproduced with the code." + chr(10)) if not_pub else ""}
{(chr(10) + (chr(10) * 2).join(rerun_notes) + chr(10)) if rerun_notes else ''}
{check_line}

Other files:

* `cv_splits.csv` — for every image (`image_id`) and fold: its role (`train`, `val` or `test`).
* `images.csv` — `image_id`, class, file name in the Mendeley archive, SHA-256 and duplicate group.
* `cv_splits_meta.json` — the duplicate-grouping rules and split settings.
* `inference.py` — loading and prediction code (below).

## Usage

```bash
pip install torch torchvision safetensors huggingface_hub pillow
```

```python
from huggingface_hub import snapshot_download
import sys
root = snapshot_download("REPO_ID")          # downloads this repository
sys.path.insert(0, root)
from inference import load_model, predict_proba, CLASSES

net, cfg = load_model("{example}", fold=0, root=root)
prob = predict_proba(net, cfg, ["leaf.jpg"])
print(CLASSES[int(prob[0].argmax())], float(prob[0].max()))
```

or from the command line: `python inference.py --model {example} --fold all leaf.jpg` (averages the five fold models).

Pre-processing (done by `inference.py`): camera orientation from EXIF, RGB, resize to 448 × 448 (bicubic), resize to the
network input size, ImageNet normalisation. Weights are stored in float16 (`.safetensors`) and loaded as float32.

**To reproduce the paper's numbers**, evaluate a fold model only on the images whose role is `test` in that fold
(`cv_splits.csv`); all other images were used to train or select that model.

## Limitations

* All images come from one public dataset, taken with one smartphone model in one region of India, under natural
  light. Accuracy on images from other regions, cultivars, devices or growing conditions is unknown and is expected
  to be lower; the reported accuracies are an upper bound for field use.
* Each image has one label. The models cannot detect several diseases on one leaf, separate early from late leaf
  spot, or grade severity. Images of other crops, other diseases or non-leaf images will still be assigned one of the
  five classes.
* These are research models. They are not validated for diagnostic or management decisions.

## Citation

The citation will be added after publication. Until then, please cite the code repository and the dataset:
Sasmal, B. *et al.* A novel groundnut leaf dataset for detection and classification of groundnut leaf diseases.
Mendeley Data, v2 (2024). https://doi.org/{DATASET_DOI}

## Licence

Weights: {HF_LICENSE.upper()}. The models were initialised from torchvision ImageNet weights (BSD-3-Clause) and
fine-tuned on the dataset above (CC BY 4.0).

## Files and checksums (SHA-256)

| File | SHA-256 |
|---|---|
''' + "\n".join(f"| `{k}` | `{v}` |" for k, v in sorted(FILES.items())) + "\n"
(STAGE / "README.md").write_text(CARD)
print(CARD[:3000])

## 5. Test the published files with `inference.py`

In [ ]:
import importlib.util, sys
sys.dont_write_bytecode = True      # no __pycache__ in the folder that is uploaded
spec = importlib.util.spec_from_file_location("hf_inference", STAGE / "inference.py")
hf_inference = importlib.util.module_from_spec(spec); spec.loader.exec_module(hf_inference)
assert hf_inference.CLASSES == CLASSES

name0 = next(iter(MODELS_TO_PUBLISH)); f0 = sorted(MODELS_TO_PUBLISH[name0]["folds"])[0]
net, ccfg = hf_inference.load_model(name0, f0, "cpu", root=STAGE)
ref = Config(**{**MODELS_TO_PUBLISH[name0]["cfg"], "pretrained": False}); ref_net = build_model(ref)
sd = torch.load(MODELS_TO_PUBLISH[name0]["folds"][f0]["weights"], map_location="cpu", weights_only=True)
ref_net.load_state_dict({k: (t.float() if t.is_floating_point() else t) for k, t in sd.items()}); ref_net.eval()
same = all(torch.equal(a, b) for a, b in zip(net.state_dict().values(), ref_net.state_dict().values()))
print(f"{name0} fold {f0}: weights loaded by inference.py identical to the original file:", same)
assert same

if VERIFY:
    s = cv[(cv["repeat"] == 0) & (cv["fold"] == f0) & (cv["role"] == "test")].head(8)
    pil = [Image.fromarray(IMAGES[i]) for i in s["image_id"]]
    p_inf = hf_inference.predict_proba(net, ccfg, pil).argmax(1).numpy()
    saved = pd.read_csv(D["results"] / name0 / f"r0_f{f0}" / "predictions.csv").set_index("image_id").loc[s["image_id"], "y_pred"].to_numpy()
    print("inference.py predictions:", p_inf.tolist(), "| reported:", saved.tolist())
    print("same predictions:", (p_inf == saved).mean() * 100, "%")

## 6. Upload to Hugging Face
Uses the `HF_TOKEN` secret (see the top of the notebook). Uploading ≈ 1.4 GB takes a few minutes from Colab.

In [ ]:
RECORD = {"env": ENV, "inventory": INVENTORY.to_dict("records"),
          "verification": {n: {str(f): r for f, r in d.items()} for n, d in VERIFICATION.items()},
          "files_sha256": FILES, "repo": None, "url": None, "commit": None, "private": HF_PRIVATE}
if UPLOAD:
    from huggingface_hub import HfApi
    token = None
    if IN_COLAB:
        from google.colab import userdata
        try:
            token = userdata.get("HF_TOKEN")
        except Exception as e:
            raise SystemExit(f"Could not read the HF_TOKEN secret ({e}). Add it in the key-icon panel and allow notebook access.")
    token = token or os.environ.get("HF_TOKEN")
    if not token:
        raise SystemExit("No HF_TOKEN found. See 'Before you run it' at the top of the notebook.")
    api = HfApi(token=token)
    user = api.whoami()["name"]
    repo_id = f"{user}/{HF_REPO_NAME}"
    card = (STAGE / "README.md").read_text().replace("REPO_ID", repo_id)
    (STAGE / "README.md").write_text(card)
    api.create_repo(repo_id, repo_type="model", private=HF_PRIVATE, exist_ok=True)
    t0 = time.time()
    info = api.upload_folder(folder_path=str(STAGE), repo_id=repo_id, repo_type="model",
                             commit_message=f"Upload models ({ENV['run_at']})",
                             ignore_patterns=["__pycache__/*", "*.pyc"])
    url = f"https://huggingface.co/{repo_id}"
    RECORD.update({"repo": repo_id, "url": url, "commit": getattr(info, "oid", None)})
    print(f"Uploaded in {(time.time()-t0)/60:.1f} min -> {url}  ({'PRIVATE' if HF_PRIVATE else 'PUBLIC'})")
else:
    print("UPLOAD = False: nothing uploaded. The folder ready for upload is", STAGE)

rec_path = D["logs"] / "hf_upload_NB06.json"
rec_path.write_text(json.dumps(RECORD, indent=2, default=str))
print("record saved:", rec_path)
print("\n==== SUMMARY (send this to me) ====")
print(INVENTORY.to_string(index=False))
if VERIFICATION:
    print(VTABLE.to_string(index=False))
print("Repository:", RECORD["url"], "| commit:", RECORD["commit"], "| private:", HF_PRIVATE)